# Laboratorio 25. Clasificación de texto: enrutamiento de tickets de clientes

**Pregunta guía:** ¿Qué ganan un modelo de bolsa de embeddings y un Transformer pequeño frente a reglas de palabras clave para enrutar tickets de clientes, y cuándo conviene derivar el ticket a una persona?

## Objetivos
Al finalizar podrás enmascarar datos personales antes de modelar; tokenizar texto en español y construir un vocabulario solo con train; entrenar una bolsa de embeddings con bigramas y un Transformer pequeño; evaluar con F1 macro y matriz de confusión; y definir un umbral de confianza que equilibre automatización y exactitud.

## Prerrequisitos y conexión
Usa embeddings (laboratorio 20), atención y Transformer (laboratorios 17 y 18). Cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Una **bolsa de embeddings** promedia los vectores de las palabras (y bigramas) del texto: ignora el orden salvo el que capturan los bigramas, pero es rápida y robusta. Un **Transformer** suma a cada token un embedding de posición y aplica auto-atención, $\text{softmax}(QK^\top/\sqrt{d})V$, de modo que cada palabra se representa en el contexto de las demás; por ejemplo, "envío" en "no es un problema de envío" puede tener otra representación que en "el envío no llegó". En ambos casos, una capa lineal final produce probabilidades por área.

## Problema y datos
Un comercio electrónico ficticio recibe tickets que deben llegar a una de cinco áreas: facturación, despacho, devolución, cuenta o garantía. Generamos 5 000 tickets con plantillas variadas, sinónimos, errores de tipeo, datos personales (correos, RUT, teléfonos), frases que mencionan otra área para descartarla y un 2 % de etiquetas erróneas. Todos los datos se generan en el notebook.

## Perspectiva de negocio y datos
Los tickets contienen datos personales que no deberían llegar a un modelo ni a sus registros si no es necesario: enmascararlos es el primer paso del *pipeline*. El objetivo de negocio no es clasificar el 100 % automáticamente, sino automatizar los casos seguros y derivar los dudosos con un nivel de exactitud acordado.

## Experimento conceptual
¿Cuánto aporta considerar el orden de las palabras? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional. Usamos `rng` (generador de NumPy) para los datos y `torch.manual_seed` antes de cada entrenamiento para que las comparaciones entre modelos sean justas.

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)

import re
import unicodedata
import zlib
from collections import Counter

**Resultado e interpretación.** La celda confirma el entorno utilizado. El laboratorio está diseñado para CPU y se ejecuta en pocos minutos; una GPU disponible se utiliza sin ser requisito.

### Generar tickets
Cada área tiene varias formas de expresar el problema. Agregamos saludos, cierres con datos personales, errores de tipeo y frases que **descartan** otra área ("no es un problema de cobro, ..."), que confunden a los métodos basados solo en palabras.

In [ ]:
class_names = ['facturacion', 'despacho', 'devolucion', 'cuenta', 'garantia']
products = ['notebook', 'celular', 'audífonos', 'refrigerador', 'zapatillas', 'chaqueta', 'televisor', 'bicicleta']
templates = {
    'facturacion': ['me cobraron {veces} la compra del {producto}', 'la boleta del pedido {num} tiene un monto incorrecto',
                    'no reconozco un cargo de {monto} pesos en mi tarjeta', 'necesito la factura de mi compra a nombre de mi empresa',
                    'me cobraron el despacho aunque era gratis', 'el descuento no se aplicó en el total que pagué'],
    'despacho': ['mi pedido {num} no ha llegado y ya pasaron {dias} días', 'el repartidor no encontró mi dirección',
                 'quiero cambiar la dirección de entrega del pedido {num}', 'el seguimiento del envío no se actualiza hace {dias} días',
                 'el paquete aparece entregado pero no lo recibí', 'la entrega del {producto} se reprogramó otra vez'],
    'devolucion': ['quiero devolver el {producto} porque no me gustó', 'compré una talla equivocada y necesito cambiarla',
                   'cómo solicito el reembolso de la compra del {producto}', 'me arrepentí de la compra del {producto}',
                   'ya envié la devolución hace {dias} días y no me devuelven el dinero', 'el {producto} no es lo que esperaba y quiero retornarlo'],
    'cuenta': ['no puedo iniciar sesión en mi cuenta', 'olvidé mi contraseña y el enlace no llega a {email}',
               'quiero cambiar el correo registrado por {email}', 'me bloquearon la cuenta después de varios intentos',
               'quiero eliminar mis datos personales de la plataforma', 'no me llega el código de verificación al {telefono}'],
    'garantia': ['el {producto} dejó de funcionar después de {meses} meses', 'el {producto} se calienta y se apaga solo',
                 'la pantalla del {producto} tiene una falla de fábrica', 'quiero hacer válida la garantía del {producto}',
                 'el {producto} hace un ruido extraño desde la primera semana', 'la batería del {producto} ya no carga'],
}
other_area_phrase = {'facturacion': 'cobro', 'despacho': 'envío', 'devolucion': 'devolución',
                     'cuenta': 'contraseña', 'garantia': 'garantía'}
greetings = ['', 'hola', 'buenas tardes', 'estimados', 'hola necesito ayuda']
closings = ['', 'gracias', 'quedo atento', 'mi rut es {rut}', 'pueden llamarme al {telefono}', 'respondan a {email}']


def random_rut():
    body = int(rng.integers(5_000_000, 25_000_000))
    return f'{body:,}'.replace(',', '.') + f'-{rng.integers(0, 10)}'


def fill(text):
    return text.format(
        producto=rng.choice(products), num=int(rng.integers(100000, 999999)), dias=int(rng.integers(2, 20)),
        meses=int(rng.integers(1, 18)), monto=int(rng.integers(5, 300)) * 1000, veces=rng.choice(['dos veces', 'doble']),
        email=f'{rng.choice(["ana", "jperez", "cliente", "mrojas"])}{rng.integers(1, 99)}@{rng.choice(["gmail.com", "correo.cl"])}',
        telefono=f'+56 9 {rng.integers(1000, 9999)} {rng.integers(1000, 9999)}', rut=random_rut())


def add_typos(text, p=0.05):
    words = text.split()
    for i, word in enumerate(words):
        if len(word) > 4 and rng.random() < p:
            j = int(rng.integers(1, len(word) - 1))
            words[i] = word[:j] + word[j + 1:]
    return ' '.join(words)


def make_ticket(label):
    body = fill(rng.choice(templates[label]))
    if rng.random() < 0.25:  # descarta explícitamente otra área
        other = rng.choice([c for c in class_names if c != label])
        body = f'no es un problema de {other_area_phrase[other]}, {body}'
    parts = [rng.choice(greetings), body, fill(rng.choice(closings))]
    return add_typos(' '.join(part for part in parts if part))


n_tickets = 5000
labels = rng.choice(5, n_tickets, p=[0.22, 0.28, 0.20, 0.15, 0.15])
texts = [make_ticket(class_names[label]) for label in labels]
noisy = rng.random(n_tickets) < 0.02  # etiquetas erróneas, como en datos reales
labels = np.where(noisy, rng.integers(0, 5, n_tickets), labels)

order = rng.permutation(n_tickets)
train_idx, val_idx, test_idx = order[:3000], order[3000:4000], order[4000:]
for i in range(4):
    print(f'[{class_names[labels[i]]}] {texts[i]}')
print('\nconteo por clase:', dict(zip(class_names, np.bincount(labels).tolist())))

**Resultado e interpretación.** Los tickets mezclan el problema con saludos, cierres y datos personales. Las frases que descartan otra área ponen en el texto palabras clave de la clase equivocada: un buen modelo debe considerar el contexto.

### Enmascarar datos personales y normalizar
Antes de construir el vocabulario reemplazamos correos, RUT, teléfonos y números por marcadores. Luego pasamos a minúsculas, quitamos tildes y separamos en tokens.

In [ ]:
PII_PATTERNS = [('<email>', re.compile(r'[\w.+-]+@[\w-]+\.[\w.]+')),
                ('<rut>', re.compile(r'\b\d{1,2}\.\d{3}\.\d{3}-[\dkK]\b')),
                ('<telefono>', re.compile(r'\+56\s?9\s?\d{4}\s?\d{4}')),
                ('<num>', re.compile(r'\b\d+\b'))]


def mask_pii(text):
    counts = {}
    for token, pattern in PII_PATTERNS:
        text, n = pattern.subn(f' {token} ', text)
        counts[token] = n
    return text, counts


def tokenize(text):
    text = unicodedata.normalize('NFKD', text.lower())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return re.findall(r'<[a-z]+>|[a-z]+', text)


masked, pii_counts = [], Counter()
for text in texts:
    clean, counts = mask_pii(text)
    masked.append(clean)
    pii_counts.update(counts)
tokens = [tokenize(text) for text in masked]
print('marcadores insertados:', dict(pii_counts))
for i in range(3):
    print(f'\noriginal:   {texts[i]}\nenmascarado: {" ".join(tokens[i])}')

**Resultado e interpretación.** Después de esta celda ningún correo, RUT o teléfono llega al vocabulario ni a los pesos del modelo. El orden importa: si se enmascara después de tokenizar, un correo ya se habría partido en fragmentos que podrían quedar en el vocabulario.

### Vocabulario y codificación
El vocabulario se construye **solo con train** (palabras con frecuencia mínima 2). Las palabras nuevas se mapean a `<unk>`. Para la bolsa de embeddings agregamos bigramas mediante *hashing* en un número fijo de cubetas.

In [ ]:
counter = Counter(token for i in train_idx for token in tokens[i])
vocab = ['<pad>', '<unk>'] + sorted(word for word, count in counter.items() if count >= 2)
word_to_id = {word: i for i, word in enumerate(vocab)}
MAX_LEN, BIGRAM_BUCKETS = 40, 2000


def encode_sequence(token_list):
    ids = [word_to_id.get(token, 1) for token in token_list][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))


def encode_bag(token_list):
    unigrams = [word_to_id.get(token, 1) for token in token_list]
    bigrams = [len(vocab) + zlib.crc32(f'{a} {b}'.encode()) % BIGRAM_BUCKETS
               for a, b in zip(token_list, token_list[1:])]
    ids = (unigrams + bigrams)[:2 * MAX_LEN]
    return ids + [0] * (2 * MAX_LEN - len(ids))


def tensors(idx, encoder):
    x = torch.tensor([encoder(tokens[i]) for i in idx], dtype=torch.long, device=device)
    return x, torch.tensor(labels[idx], dtype=torch.long, device=device)


seq_data = {name: tensors(idx, encode_sequence) for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]}
bag_data = {name: tensors(idx, encode_bag) for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]}
unknown_rate = np.mean([word_to_id.get(t, 1) == 1 for i in val_idx for t in tokens[i]])
print('tamaño del vocabulario:', len(vocab), '| tokens desconocidos en validación:', round(float(unknown_rate), 3))

**Resultado e interpretación.** La proporción de tokens desconocidos en validación proviene sobre todo de los errores de tipeo. Construir el vocabulario con todos los datos haría parecer que ese problema no existe: sería una fuga de información de validación y test.

### Baseline de palabras clave
Una regla cuenta palabras clave por área y elige la de mayor puntaje (o la clase más frecuente si no hay coincidencias). Es la solución típica antes de un modelo.

In [ ]:
keywords = {'facturacion': ['cobr', 'cargo', 'boleta', 'factura', 'monto', 'descuento', 'pague'],
            'despacho': ['llegado', 'repartidor', 'direccion', 'envio', 'seguimiento', 'paquete', 'entrega'],
            'devolucion': ['devolver', 'devolucion', 'reembolso', 'talla', 'arrepenti', 'retornar'],
            'cuenta': ['sesion', 'contrasena', 'cuenta', 'correo', 'bloquearon', 'codigo'],
            'garantia': ['funcionar', 'falla', 'garantia', 'apaga', 'ruido', 'bateria', 'pantalla']}
majority = int(np.bincount(labels[train_idx]).argmax())


def keyword_rule(token_list):
    text = ' '.join(token_list)
    scores = [sum(text.count(word) for word in keywords[name]) for name in class_names]
    return int(np.argmax(scores)) if max(scores) > 0 else majority


def macro_f1(y_true, y_pred, n_classes=5):
    f1 = []
    for c in range(n_classes):
        tp = ((y_pred == c) & (y_true == c)).sum()
        precision = tp / max((y_pred == c).sum(), 1)
        recall = tp / max((y_true == c).sum(), 1)
        f1.append(2 * precision * recall / max(precision + recall, 1e-9))
    return float(np.mean(f1))


rule_pred = np.array([keyword_rule(tokens[i]) for i in val_idx])
print('reglas en validación: exactitud', round(float((rule_pred == labels[val_idx]).mean()), 4),
      '| F1 macro', round(macro_f1(labels[val_idx], rule_pred), 4))

**Resultado e interpretación.** Las reglas funcionan razonablemente cuando el cliente usa las palabras esperadas, pero fallan con sinónimos, errores de tipeo y frases que mencionan otra área. Además, cada regla nueva hay que mantenerla a mano.

### Bolsa de embeddings y Transformer pequeño
`nn.EmbeddingBag` promedia embeddings ignorando el relleno. El Transformer usa embeddings de token y de posición, dos capas de auto-atención con máscara de relleno y un promedio de las representaciones de los tokens reales.

In [ ]:
class BagOfEmbeddings(nn.Module):
    def __init__(self, n_embeddings, dim=32, n_classes=5):
        super().__init__()
        self.bag = nn.EmbeddingBag(n_embeddings, dim, mode='mean', padding_idx=0)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(dim, n_classes))

    def forward(self, x):
        return self.classifier(self.bag(x))


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, dim=32, heads=4, layers=2, n_classes=5):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.position_embedding = nn.Embedding(MAX_LEN, dim)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=64, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers, enable_nested_tensor=False)
        self.classifier = nn.Linear(dim, n_classes)

    def forward(self, x):
        padding = x == 0
        positions = torch.arange(x.shape[1], device=x.device)
        h = self.encoder(self.token_embedding(x) + self.position_embedding(positions), src_key_padding_mask=padding)
        keep = (~padding).unsqueeze(-1).float()
        return self.classifier((h * keep).sum(dim=1) / keep.sum(dim=1).clamp(min=1.0))


builders = {'bolsa_embeddings': (lambda: BagOfEmbeddings(len(vocab) + BIGRAM_BUCKETS), bag_data, 3e-3),
            'transformer': (lambda: TinyTransformer(len(vocab)), seq_data, 1e-3)}
for name, (build, _, _) in builders.items():
    print(f'{name:17s} parámetros: {sum(p.numel() for p in build().parameters())}')

**Resultado e interpretación.** La bolsa de embeddings concentra casi todos sus parámetros en la tabla de embeddings; el Transformer agrega parámetros de atención y capas *feed-forward*. La comparación muestra si el contexto que captura la atención compensa su costo.

### Entrenar ambos modelos
Mismo protocolo: semilla, *early stopping* según la pérdida de validación y batches de 64.

In [ ]:
def train_text_model(build_model, data, lr, epochs=40, batch_size=64, patience=6):
    torch.manual_seed(SEED)
    model = build_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(*data['train']), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    curve, best_loss, best_state, wait = [], np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for x, target in loader:
            loss = F.cross_entropy(model(x), target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(data['val'][0]), data['val'][1]).item()
        curve.append(val_loss)
        if val_loss < best_loss - 1e-4:
            best_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, curve


def predict_proba(model, data, split):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(data[split][0]), dim=1).cpu().numpy()


text_models, curves = {}, {}
for name, (build, data, lr) in builders.items():
    start = time.perf_counter()
    text_models[name], curves[name] = train_text_model(build, data, lr)
    print(f'{name:17s} {len(curves[name])} épocas | {time.perf_counter() - start:.1f} s')
    plt.plot(curves[name], label=name)
plt.xlabel('época')
plt.ylabel('entropía cruzada (validación)')
plt.legend()
plt.show()

**Resultado e interpretación.** Compara la pérdida mínima de validación y el tiempo de entrenamiento. Con 2 % de etiquetas erróneas, la pérdida no puede llegar a cero: parte del error es ruido de etiquetado y no del modelo.

### Comparar y analizar errores
Comparamos reglas y modelos en validación, elegimos por F1 macro y revisamos ejemplos mal clasificados del modelo elegido.

In [ ]:
probabilities = {name: predict_proba(model, builders[name][1], 'val') for name, model in text_models.items()}
results = {'reglas': rule_pred, **{name: p.argmax(1) for name, p in probabilities.items()}}
for name, pred in results.items():
    print(f'{name:17s} validación: exactitud {(pred == labels[val_idx]).mean():.4f} | '
          f'F1 macro {macro_f1(labels[val_idx], pred):.4f}')

chosen = max(text_models, key=lambda name: macro_f1(labels[val_idx], results[name]))
test_proba = predict_proba(text_models[chosen], builders[chosen][1], 'test')
test_pred = test_proba.argmax(1)
print(f'\nElegido: {chosen} | test: exactitud {(test_pred == labels[test_idx]).mean():.4f} | '
      f'F1 macro {macro_f1(labels[test_idx], test_pred):.4f}')

print('\nEjemplos mal clasificados en validación:')
wrong = np.flatnonzero(results[chosen] != labels[val_idx])[:5]
for i in wrong:
    print(f'  real={class_names[labels[val_idx][i]]:11s} pred={class_names[results[chosen][i]]:11s} | {masked[val_idx[i]].strip()}')

**Resultado e interpretación.** Lee los errores antes de concluir: algunos son etiquetas erróneas del propio dataset (el modelo acierta y la etiqueta no), otros son frases que descartan un área y otros son tickets realmente ambiguos. Esa clasificación de errores orienta qué mejorar: datos, etiquetas o modelo.

### Automatizar solo los casos seguros
Enrutamos automáticamente si la probabilidad máxima supera un umbral $\tau$; si no, el ticket va a una persona. En validación elegimos el menor $\tau$ que alcanza una exactitud automática de 97 %, y lo aplicamos en test.

In [ ]:
TARGET_ACCURACY = 0.97
val_proba = probabilities[chosen]
val_confidence, val_pred = val_proba.max(axis=1), val_proba.argmax(axis=1)
thresholds = np.linspace(0.30, 0.99, 70)
coverage, accuracy = [], []
for t in thresholds:
    automatic = val_confidence >= t
    coverage.append(automatic.mean())
    accuracy.append((val_pred[automatic] == labels[val_idx][automatic]).mean() if automatic.any() else np.nan)
valid = [t for t, a in zip(thresholds, accuracy) if a >= TARGET_ACCURACY]
tau = float(min(valid)) if valid else float(thresholds[-1])

automatic = test_proba.max(axis=1) >= tau
print(f'umbral elegido en validación: {tau:.2f}')
print(f'test: {automatic.mean():.1%} de tickets automatizados con exactitud '
      f'{(test_pred[automatic] == labels[test_idx][automatic]).mean():.4f}; {(~automatic).sum()} derivados a personas')

plt.plot(coverage, accuracy, marker='.')
plt.axhline(TARGET_ACCURACY, color='gray', linestyle='--')
plt.xlabel('fracción automatizada (cobertura)')
plt.ylabel('exactitud de lo automatizado')
plt.title('Validación: cobertura frente a exactitud')
plt.show()

**Resultado e interpretación.** La curva muestra el *trade-off*: automatizar más implica aceptar más errores. El umbral convierte una meta de servicio (97 % de exactitud) en una regla operativa, y el resto de los tickets sigue un flujo humano. Si las probabilidades del modelo están mal calibradas, el umbral elegido en validación puede no cumplir la meta en test: verifícalo siempre.

## Limitaciones y conclusiones clave
Los tickets se generaron con plantillas y por eso son relativamente fáciles: con estas frases, una bolsa de embeddings con bigramas ya se acerca al techo que impone el 2 % de etiquetas erróneas, y es esperable que el Transformer no la supere. Su ventaja aparece con textos más largos y variados, donde el contexto y el orden importan más. El lenguaje real es más variado, con jerga, mensajes largos y varios problemas en un mismo ticket. En la práctica se parte de modelos de lenguaje preentrenados (o de un LLM con salida estructurada) y se ajusta con datos etiquetados; los principios de este laboratorio (enmascarar PII, vocabulario solo de train, umbral de confianza, análisis de errores) se mantienen.

- El enmascaramiento de datos personales es un paso del *pipeline*, previo al vocabulario.
- La bolsa de embeddings es una referencia fuerte y barata; el Transformer agrega contexto y orden.
- La decisión operativa combina el modelo con un umbral de confianza y un flujo humano para los casos dudosos.

## Ejercicios progresivos
1. **Guiado:** elimina los bigramas de `encode_bag` (solo unigramas), reentrena la bolsa de embeddings y compara el F1 macro en los tickets que contienen "no es un problema de".
2. **Independiente:** sube la probabilidad de errores de tipeo a 0,15, regenera los datos y compara cuánto cae cada modelo. Propón una solución basada en subpalabras o n-gramas de caracteres.
3. **Desafío:** calibra las probabilidades del modelo elegido con *temperature scaling* (un escalar $T$ ajustado en validación que divide los logits) y verifica si el umbral elegido cumple mejor la meta de exactitud en test.

## Conexión con el siguiente laboratorio
Los n-gramas de caracteres también sirven para comparar registros con errores de tipeo. En el laboratorio 26 los usamos para detectar clientes duplicados.